# 1. Data Preprocessing

In [21]:
import pandas as pd
import numpy as np
from pgmpy.models import BayesianNetwork
from pgmpy.estimators import MaximumLikelihoodEstimator
from pgmpy.inference import VariableElimination
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, roc_auc_score, balanced_accuracy_score
from sklearn.gaussian_process import GaussianProcessClassifier
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.pipeline import make_pipeline
from scipy.stats import entropy  # For KL Divergence
import time

# Load and preprocess the dementia dataset
data = pd.read_csv('dementia_data-MRI-features.csv')
data['SES'].fillna(data['SES'].median(), inplace=True)
data['Group'] = data['Group'].map({'Nondemented': 0, 'Demented': 1})
data.dropna(subset=['Group'], inplace=True)
data['MMSE'].fillna(data['MMSE'].median(), inplace=True)

# Convert any categorical data to numerical values
for column in data.select_dtypes(include=['object']).columns:
    le = LabelEncoder()
    data[column] = le.fit_transform(data[column])

# Define features and target
X = data.drop(columns=['Subject ID', 'MRI ID', 'Group'])
y = data['Group']

# Split data into training and test sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


# 2. Discrete Bayesian Network Model

In [22]:
# Define and train the Bayesian Network
discrete_model = BayesianNetwork([
    ('Age', 'Group'),
    ('EDUC', 'Group'),
    ('MMSE', 'Group'),
    ('CDR', 'Group'),
    ('SES', 'Group'),
])

# Train the Discrete Bayesian Network and time it
start_time = time.time()
discrete_model.fit(data, estimator=MaximumLikelihoodEstimator)
train_time_discrete = time.time() - start_time

# Inference using Variable Elimination
discrete_inference = VariableElimination(discrete_model)

# Make predictions for the test set and time the inference
discrete_predictions = []
discrete_proba = []
start_time = time.time()
for _, row in X_test.iterrows():
    evidence = {
        'Age': row['Age'],
        'EDUC': row['EDUC'],
        'MMSE': row['MMSE'],
        'CDR': row['CDR'],
        'SES': row['SES']
    }
    pred = discrete_inference.map_query(variables=['Group'], evidence=evidence)
    prob = discrete_inference.query(variables=['Group'], evidence=evidence).values
    discrete_predictions.append(pred['Group'])
    discrete_proba.append(prob[1])  # Probability of the positive class
test_time_discrete = time.time() - start_time


0it [00:00, ?it/s]

0it [00:00, ?it/s]

0it [00:00, ?it/s]

0it [00:00, ?it/s]

0it [00:00, ?it/s]

0it [00:00, ?it/s]

... (126 outputs left out)

0it [00:00, ?it/s]

0it [00:00, ?it/s]

0it [00:00, ?it/s]

0it [00:00, ?it/s]

# 3. Discrete Model Metrics Calculation

In [23]:
# Calculate metrics for Discrete Bayesian Network
discrete_accuracy = accuracy_score(y_test, discrete_predictions)
discrete_auc = roc_auc_score(y_test, discrete_proba)
discrete_balanced_accuracy = balanced_accuracy_score(y_test, discrete_predictions)
discrete_kl_divergence = entropy(y_test, discrete_proba)

print(f"Discrete Bayesian Network - Accuracy: {discrete_accuracy:.2f}, AUC: {discrete_auc:.2f}, "
      f"Balanced Accuracy: {discrete_balanced_accuracy:.2f}, KL Divergence: {discrete_kl_divergence:.2f}")
print(f"Discrete Bayesian Network - Training Time: {train_time_discrete:.2f}s, Testing Time: {test_time_discrete:.2f}s")


Discrete Bayesian Network - Accuracy: 1.00, AUC: 1.00, Balanced Accuracy: 1.00, KL Divergence: 0.00
Discrete Bayesian Network - Training Time: 17.24s, Testing Time: 10.44s


# 4. Gaussian Bayesian Network Model

In [24]:
# Scale the data for Gaussian Process
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Define and fit the Gaussian Process model and time it
gaussian_model = make_pipeline(StandardScaler(), GaussianProcessClassifier())
start_time = time.time()
gaussian_model.fit(X_train_scaled, y_train)
train_time_gaussian = time.time() - start_time

# Make predictions and calculate probabilities for Gaussian Model, and time the inference
start_time = time.time()
gaussian_predictions = gaussian_model.predict(X_test_scaled)
gaussian_proba = gaussian_model.predict_proba(X_test_scaled)[:, 1]  # Probability of the positive class
test_time_gaussian = time.time() - start_time


# 5. Gaussian Model Metrics Calculation

In [25]:
# Calculate metrics for Gaussian Process Model
gaussian_accuracy = accuracy_score(y_test, gaussian_predictions)
gaussian_auc = roc_auc_score(y_test, gaussian_proba)
gaussian_balanced_accuracy = balanced_accuracy_score(y_test, gaussian_predictions)
gaussian_kl_divergence = entropy(y_test, gaussian_proba)

print(f"Gaussian Bayesian Network - Accuracy: {gaussian_accuracy:.2f}, AUC: {gaussian_auc:.2f}, "
      f"Balanced Accuracy: {gaussian_balanced_accuracy:.2f}, KL Divergence: {gaussian_kl_divergence:.2f}")
print(f"Gaussian Bayesian Network - Training Time: {train_time_gaussian:.2f}s, Testing Time: {test_time_gaussian:.2f}s")


Gaussian Bayesian Network - Accuracy: 1.00, AUC: 1.00, Balanced Accuracy: 1.00, KL Divergence: 0.61
Gaussian Bayesian Network - Training Time: 0.03s, Testing Time: 0.00s


# 6. Comparison of Results

In [26]:
# Comparison of Results
print("\nComparison of Discrete and Gaussian Bayesian Networks:")
print(f"Discrete Model - Accuracy: {discrete_accuracy:.2f}, AUC: {discrete_auc:.2f}, "
      f"Balanced Accuracy: {discrete_balanced_accuracy:.2f}, KL Divergence: {discrete_kl_divergence:.2f}, "
      f"Training Time: {train_time_discrete:.2f}s, Testing Time: {test_time_discrete:.2f}s")

print(f"Gaussian Model - Accuracy: {gaussian_accuracy:.2f}, AUC: {gaussian_auc:.2f}, "
      f"Balanced Accuracy: {gaussian_balanced_accuracy:.2f}, KL Divergence: {gaussian_kl_divergence:.2f}, "
      f"Training Time: {train_time_gaussian:.2f}s, Testing Time: {test_time_gaussian:.2f}s")



Comparison of Discrete and Gaussian Bayesian Networks:
Discrete Model - Accuracy: 1.00, AUC: 1.00, Balanced Accuracy: 1.00, KL Divergence: 0.00, Training Time: 17.24s, Testing Time: 10.44s
Gaussian Model - Accuracy: 1.00, AUC: 1.00, Balanced Accuracy: 1.00, KL Divergence: 0.61, Training Time: 0.03s, Testing Time: 0.00s
